In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def classify_error(rmse_normalized):
    if rmse_normalized < 0.1:
        return 'baixo'
    elif rmse_normalized < 0.3:
        return 'médio'
    else:
        return 'alto'


In [ ]:
pasta_prediction = '../../results/regression/predictions-regression-bysource'

In [ ]:
import os
import pandas as pd

# Função para classificar valores nas faixas
def substituir_valor(valor):
    if valor < 200000000:
        return "r"
    elif 200000000 <= valor < 500000000:
        return "o"
    elif 500000000 <= valor < 800000000:
        return "y"
    elif 800000000 <= valor < 1000000000:
        return "b"
    else:
        return "g"

# Caminho da pasta onde estão os arquivos CSV
caminho_pasta = pasta_prediction

# Listar os arquivos da pasta
arquivos = os.listdir(caminho_pasta)

# Inicializar lista para armazenar os resultados
resultados = []

# Iterar sobre os arquivos
for arquivo in arquivos:
    if arquivo.endswith(".csv") and arquivo.startswith("Vazao_bbr"):
        # Caminho completo do arquivo
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        
        # Carregar o arquivo CSV
        df = pd.read_csv(caminho_arquivo)
        
        # Verificar se a coluna 'y_test' está no DataFrame
        if 'y_test' in df.columns:
            # Classificar os valores de y_test nas faixas
            df['faixa_y_test'] = df['y_test'].apply(substituir_valor)
            
            # Iterar sobre as previsões de cada modelo
            for modelo in df.columns:
                if modelo.startswith('y_predict'):  # Ignorar a coluna 'y_test'
                    # Classificar os valores das previsões nas faixas
                    coluna_faixa = f'faixa_{modelo}'
                    df[coluna_faixa] = df[modelo].apply(substituir_valor)
                    
                    # Verificar se y_test e y_pred estão na mesma faixa
                    coluna_comparacao = f'mesma_faixa_{modelo}'
                    df[coluna_comparacao] = df['faixa_y_test'] == df[coluna_faixa]
                    
                    # Contabilizar quantos estão na mesma faixa
                    mesma_faixa_count = df[coluna_comparacao].sum()
                    total_count = df.shape[0]
                    
                    # Armazenar os resultados para cada modelo
                    resultados.append({
                        "arquivo": arquivo.split("_")[2].split(".")[0],# + " "+arquivo.split("_")[1],
                        "modelo": modelo,
                        "mesma_faixa_count": mesma_faixa_count,
                        "total_count": total_count,
                        "percentual_misma_faixa": (mesma_faixa_count / total_count) * 100
                    })

# Converter resultados para um DataFrame
df_resultados = pd.DataFrame(resultados)



In [ ]:
df_resultados

In [ ]:
dropado = df_resultados.drop(columns = ['arquivo', 'modelo'])

In [ ]:
mean = dropado.mean()

In [ ]:
mean

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar o arquivo CSV
df = df_resultados
# Configurar o estilo do gráfico
sns.set(style="whitegrid")

# Plotar o gráfico de barras
plt.figure(figsize=(12, 6))

# Criar o gráfico de barras agrupado por 'arquivo' (source) e os modelos
sns.barplot(data=df, x='arquivo', y='percentual_misma_faixa', hue='modelo', ci=None)

# Título e rótulos do gráfico
plt.title('Percentual de Previsões na Mesma Faixa por Fonte (Source) e Modelo', fontsize=14)
plt.xlabel('Fonte (Source)', fontsize=12)
plt.ylabel('Percentual na Mesma Faixa (%)', fontsize=12)

# Rotacionar os rótulos do eixo X para melhor visualização
plt.xticks(rotation=0)

# Exibir a legenda
plt.legend(title='Modelo', bbox_to_anchor=(1.05, 1), loc='upper left')

# Exibir o gráfico
plt.tight_layout()
plt.show()
